## **Case Study: From Data to Diagnosis: Handling Missing Values and Aggregating Patient Records with Pandas**

Business Problem
A hospital wants to improve patient care and operational efficiency.
They need to analyze patient encounter data to:
** Identify trends in readmissions and costs
** Filter and segment patients by encounter type and payer
** Handle missing or incomplete records
** Aggregate costs and stay durations by patient groups
** Sort and merge information for actionable insights

In [1]:
import pandas as pd

In [2]:
df = pd.read_csv('encounters.csv')
df.head()

,Id,START,STOP,PATIENT,ORGANIZATION,PAYER,ENCOUNTERCLASS,CODE,DESCRIPTION,BASE_ENCOUNTER_COST,TOTAL_CLAIM_COST,PAYER_COVERAGE,REASONCODE,REASONDESCRIPTION
0,32c84703-2481-49cd-d571-3899d5820253,2011-01-02T09:26:36Z,2011-01-02T12:58:36Z,3de74169-7f67-9304-91d4-757e0f3a14d2,d78e84ec-30aa-3bba-a33a-f29a3a454662,b1c428d6-4f07-31e0-90f0-68ffa6ff8c76,ambulatory,185347001,Encounter for problem (procedure),85.55,1018.02,0.00,NaN,NaN
1,c98059da-320a-c0a6-fced-c8815f3e3f39,2011-01-03T05:44:39Z,2011-01-03T06:01:42Z,d9ec2e44-32e9-9148-179a-1653348cc4e2,d78e84ec-30aa-3bba-a33a-f29a3a454662,b1c428d6-4f07-31e0-90f0-68ffa6ff8c76,outpatient,308335008,Patient encounter procedure,142.58,2619.36,0.00,NaN,NaN
2,4ad28a3a-2479-782b-f29c-d5b3f41a001e,2011-01-03T14:32:11Z,2011-01-03T14:47:11Z,73babadf-5b2b-fee7-189e-6f41ff213e01,d78e84ec-30aa-3bba-a33a-f29a3a454662,7caa7254-5050-3b5e-9eae-bd5ea30e809c,outpatient,185349003,Encounter for check up (procedure),85.55,461.59,305.27,NaN,NaN
3,c3f4da61-e4b4-21d5-587a-fbc89943bc19,2011-01-03T16:24:45Z,2011-01-03T16:39:45Z,3b46a0b7-0f34-9b9a-c319-ace4a1f58c0b,d78e84ec-30aa-3bba-a33a-f29a3a454662,b1c428d6-4f07-31e0-90f0-68ffa6ff8c76,wellness,162673000,General examination of patient (procedure),136.80,1784.24,0.00,NaN,NaN
4,a9183b4f-2572-72ea-54c2-b3cd038b4be7,2011-01-03T17:36:53Z,2011-01-03T17:51:53Z,fa006887-d93c-d302-8b89-f3c25f88c0e1,d78e84ec-30aa-3bba-a33a-f29a3a454662,42c4fca7-f8a9-3cd1-982a-dd9751bf3e2a,ambulatory,390906007,Follow-up encounter,85.55,234.72,0.00,55822004.0,Hyperlipidemia


2. Column Selection & Filtering


In [3]:
# Select columns of interest
possible_cols = ['Id', 'PATIENT', 'ENCOUNTERCLASS', 'BASE_ENCOUNTER_COST', 'PAYER', 'REASONDESCRIPTION']
possible_cols_lower = [c.lower() for c in possible_cols]

In [4]:
# Find which columns exist

selected_cols = [c for c in df.columns if c.upper() in [p.upper() for p in possible_cols]]
if not selected_cols:
    selected_cols = [c for c in df.columns if c.lower() in possible_cols_lower]

print("Selected columns:", selected_cols)
selected = df[selected_cols]
selected.head()

Selected columns: ['Id', 'PATIENT', 'PAYER', 'ENCOUNTERCLASS', 'BASE_ENCOUNTER_COST', 'REASONDESCRIPTION']


,Id,PATIENT,PAYER,ENCOUNTERCLASS,BASE_ENCOUNTER_COST,REASONDESCRIPTION
0,32c84703-2481-49cd-d571-3899d5820253,3de74169-7f67-9304-91d4-757e0f3a14d2,b1c428d6-4f07-31e0-90f0-68ffa6ff8c76,ambulatory,85.55,NaN
1,c98059da-320a-c0a6-fced-c8815f3e3f39,d9ec2e44-32e9-9148-179a-1653348cc4e2,b1c428d6-4f07-31e0-90f0-68ffa6ff8c76,outpatient,142.58,NaN
2,4ad28a3a-2479-782b-f29c-d5b3f41a001e,73babadf-5b2b-fee7-189e-6f41ff213e01,7caa7254-5050-3b5e-9eae-bd5ea30e809c,outpatient,85.55,NaN
3,c3f4da61-e4b4-21d5-587a-fbc89943bc19,3b46a0b7-0f34-9b9a-c319-ace4a1f58c0b,b1c428d6-4f07-31e0-90f0-68ffa6ff8c76,wellness,136.80,NaN
4,a9183b4f-2572-72ea-54c2-b3cd038b4be7,fa006887-d93c-d302-8b89-f3c25f88c0e1,42c4fca7-f8a9-3cd1-982a-dd9751bf3e2a,ambulatory,85.55,Hyperlipidemia


### 3. Handling Missing Values


In [5]:
print("\nMissing values in each column:")
df.isnull().sum()


Missing values in each column:


Id                         0
START                      0
STOP                       0
PATIENT                    0
ORGANIZATION               0
PAYER                      0
ENCOUNTERCLASS             0
CODE                       0
DESCRIPTION                0
BASE_ENCOUNTER_COST        0
TOTAL_CLAIM_COST           0
PAYER_COVERAGE             0
REASONCODE             19541
REASONDESCRIPTION      19541
dtype: int64

**Fill missing cost values with the median**

In [6]:
df['BASE_ENCOUNTER_COST'].fillna(df['BASE_ENCOUNTER_COST'].median(), inplace=True)

C:\Users\navne\AppData\Local\Temp\ipykernel_23852\3107358296.py:1: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  df['BASE_ENCOUNTER_COST'].fillna(df['BASE_ENCOUNTER_COST'].median(), inplace=True)


0         85.55
1        142.58
2         85.55
3        136.80
4         85.55
          ...  
27886     85.55
27887    142.58
27888    142.58
27889    136.80
27890    136.80
Name: BASE_ENCOUNTER_COST, Length: 27891, dtype: float64

**Drop rows where 'ReasonDescription' is missing**

In [7]:
df.dropna(subset=['REASONDESCRIPTION'], inplace=True)
print("\nAfter handling missing values:")
df.head()


After handling missing values:


,Id,START,STOP,PATIENT,ORGANIZATION,PAYER,ENCOUNTERCLASS,CODE,DESCRIPTION,BASE_ENCOUNTER_COST,TOTAL_CLAIM_COST,PAYER_COVERAGE,REASONCODE,REASONDESCRIPTION
4,a9183b4f-2572-72ea-54c2-b3cd038b4be7,2011-01-03T17:36:53Z,2011-01-03T17:51:53Z,fa006887-d93c-d302-8b89-f3c25f88c0e1,d78e84ec-30aa-3bba-a33a-f29a3a454662,42c4fca7-f8a9-3cd1-982a-dd9751bf3e2a,ambulatory,390906007,Follow-up encounter,85.55,234.72,0.00,55822004.0,Hyperlipidemia
7,2cfd4ddd-ad13-fe1e-528b-15051cea2ec3,2011-01-04T14:49:55Z,2011-01-04T15:04:55Z,d856d6e6-4c98-e7a2-129b-44076c63d008,d78e84ec-30aa-3bba-a33a-f29a3a454662,7c4411ce-02f1-39b5-b9ec-dfbea9ad3c1a,ambulatory,185347001,Encounter for problem,85.55,11855.19,11205.43,363406005.0,Malignant tumor of colon
9,17966936-0878-f4db-128b-a43ae10d0878,2011-01-05T04:02:09Z,2011-01-05T04:17:09Z,bc9d59c3-0a30-6e3b-f47d-022e4f03c8de,d78e84ec-30aa-3bba-a33a-f29a3a454662,7caa7254-5050-3b5e-9eae-bd5ea30e809c,outpatient,185347001,Encounter for problem,85.55,9881.17,7872.94,254637007.0,Non-small cell lung cancer (disorder)
14,4b6a38e6-7df9-2d75-d850-6bd662f3d05b,2011-01-07T21:02:09Z,2011-01-07T22:39:45Z,bc9d59c3-0a30-6e3b-f47d-022e4f03c8de,d78e84ec-30aa-3bba-a33a-f29a3a454662,7caa7254-5050-3b5e-9eae-bd5ea30e809c,outpatient,185347001,Encounter for problem,85.55,5136.00,4076.80,424132000.0,Non-small cell carcinoma of lung TNM stage 1 ...
16,f39e72a6-a58e-0aea-cc1f-8aee6f126cc0,2011-01-08T15:20:55Z,2011-01-08T15:35:55Z,88986fec-6364-262e-37d4-e20ceb511166,d78e84ec-30aa-3bba-a33a-f29a3a454662,7caa7254-5050-3b5e-9eae-bd5ea30e809c,ambulatory,390906007,Follow-up encounter,85.55,234.72,155.77,55822004.0,Hyperlipidemia


In [8]:
print("\nMissing values in each column:")
df.isnull().sum()


Missing values in each column:


Id                     0
START                  0
STOP                   0
PATIENT                0
ORGANIZATION           0
PAYER                  0
ENCOUNTERCLASS         0
CODE                   0
DESCRIPTION            0
BASE_ENCOUNTER_COST    0
TOTAL_CLAIM_COST       0
PAYER_COVERAGE         0
REASONCODE             0
REASONDESCRIPTION      0
dtype: int64

### **4. Aggregations (groupby, mean)**

**Average cost per encounter class**

In [9]:
avg_cost_by_class = df.groupby('ENCOUNTERCLASS')['BASE_ENCOUNTER_COST'].mean()
print("\nAverage Base Encounter Cost by Encounter Class:")
print(avg_cost_by_class)



Average Base Encounter Cost by Encounter Class:
ENCOUNTERCLASS
ambulatory    110.034084
emergency     139.201323
inpatient     135.808129
outpatient     85.550000
wellness      136.800000
Name: BASE_ENCOUNTER_COST, dtype: float64


**Total claim cost by payer**

In [10]:
total_claim_by_payer = df.groupby('PAYER')['TOTAL_CLAIM_COST'].sum()
print("\nTotal Claim Cost by Payer:")
total_claim_by_payer


Total Claim Cost by Payer:


PAYER
047f6ec3-6215-35eb-9608-f9dda363a44c    1195513.62
42c4fca7-f8a9-3cd1-982a-dd9751bf3e2a    1700220.24
4d71f845-a6a9-3c39-b242-14d25ef86a8d    1091943.15
5059a55e-5d6e-34d1-b6cb-d83d16e57bcf    1051841.75
6e2f1a2d-27bd-3701-8d08-dae202c58632    1623290.20
7c4411ce-02f1-39b5-b9ec-dfbea9ad3c1a    7165889.64
7caa7254-5050-3b5e-9eae-bd5ea30e809c    2828182.83
b1c428d6-4f07-31e0-90f0-68ffa6ff8c76    9983932.33
b3221cfc-24fb-339e-823d-bc4136cbc4ed     534523.99
d47b3510-2895-3b70-9897-342d681c769d    1354417.07
Name: TOTAL_CLAIM_COST, dtype: float64

### 5. Sorting, Merging, Apply

In [11]:
df.head()

,Id,START,STOP,PATIENT,ORGANIZATION,PAYER,ENCOUNTERCLASS,CODE,DESCRIPTION,BASE_ENCOUNTER_COST,TOTAL_CLAIM_COST,PAYER_COVERAGE,REASONCODE,REASONDESCRIPTION
4,a9183b4f-2572-72ea-54c2-b3cd038b4be7,2011-01-03T17:36:53Z,2011-01-03T17:51:53Z,fa006887-d93c-d302-8b89-f3c25f88c0e1,d78e84ec-30aa-3bba-a33a-f29a3a454662,42c4fca7-f8a9-3cd1-982a-dd9751bf3e2a,ambulatory,390906007,Follow-up encounter,85.55,234.72,0.00,55822004.0,Hyperlipidemia
7,2cfd4ddd-ad13-fe1e-528b-15051cea2ec3,2011-01-04T14:49:55Z,2011-01-04T15:04:55Z,d856d6e6-4c98-e7a2-129b-44076c63d008,d78e84ec-30aa-3bba-a33a-f29a3a454662,7c4411ce-02f1-39b5-b9ec-dfbea9ad3c1a,ambulatory,185347001,Encounter for problem,85.55,11855.19,11205.43,363406005.0,Malignant tumor of colon
9,17966936-0878-f4db-128b-a43ae10d0878,2011-01-05T04:02:09Z,2011-01-05T04:17:09Z,bc9d59c3-0a30-6e3b-f47d-022e4f03c8de,d78e84ec-30aa-3bba-a33a-f29a3a454662,7caa7254-5050-3b5e-9eae-bd5ea30e809c,outpatient,185347001,Encounter for problem,85.55,9881.17,7872.94,254637007.0,Non-small cell lung cancer (disorder)
14,4b6a38e6-7df9-2d75-d850-6bd662f3d05b,2011-01-07T21:02:09Z,2011-01-07T22:39:45Z,bc9d59c3-0a30-6e3b-f47d-022e4f03c8de,d78e84ec-30aa-3bba-a33a-f29a3a454662,7caa7254-5050-3b5e-9eae-bd5ea30e809c,outpatient,185347001,Encounter for problem,85.55,5136.00,4076.80,424132000.0,Non-small cell carcinoma of lung TNM stage 1 ...
16,f39e72a6-a58e-0aea-cc1f-8aee6f126cc0,2011-01-08T15:20:55Z,2011-01-08T15:35:55Z,88986fec-6364-262e-37d4-e20ceb511166,d78e84ec-30aa-3bba-a33a-f29a3a454662,7caa7254-5050-3b5e-9eae-bd5ea30e809c,ambulatory,390906007,Follow-up encounter,85.55,234.72,155.77,55822004.0,Hyperlipidemia


In [12]:
# Sort encounters by highest base cost
sorted_df = df.sort_values(by='BASE_ENCOUNTER_COST', ascending=False)
print("\nTop 5 most expensive encounters:")
sorted_df[['Id', 'BASE_ENCOUNTER_COST']].head()


Top 5 most expensive encounters:


,Id,BASE_ENCOUNTER_COST
95,7929bcef-0b8c-cbc9-83fd-dbaf57400f7d,146.18
13550,595809db-e889-3b91-26f1-8ff44e87f71c,146.18
13635,db23cdf6-4ae1-0413-6fea-f40cf5f70787,146.18
13633,87d077d7-41cc-16e0-ae35-bf142060f2ea,146.18
15051,b422dab9-9801-64ed-0124-10a219d1ac5c,146.18
